# MiWay GTFS Route Efficiency Project - Notebook 5

## Final route efficiency ranking

This notebook combines the outputs from Notebooks 2, 3, and 4 into one final route-level ranking for **Tuesday, May 5, 2026**.

The final score combines:

1. scheduled speed,
2. frequency,
3. service span,
4. route directness,
5. stop-spacing efficiency,
6. directional balance.

The goal is not to declare that one route is objectively good or bad. The goal is to create a transparent, repeatable first-pass ranking that can support advocacy questions about where service is strong, where riders may face delays, and which routes deserve closer review.

## 1. Import libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 160)

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print('matplotlib is not installed, so chart cells will be skipped.')

## 2. Locate project folders and required outputs

In [ ]:
candidate_roots = [Path.cwd(), Path.cwd().parent]

PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'Outputs').exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project Outputs folder from this notebook location.')

OUTPUT_DIR = PROJECT_ROOT / 'Outputs'

GEOMETRY_PATH = OUTPUT_DIR / 'route_geometry_stop_metrics_2026-05-05.csv'
SPEED_PATH = OUTPUT_DIR / 'route_scheduled_speed_metrics_2026-05-05.csv'
FREQUENCY_PATH = OUTPUT_DIR / 'route_frequency_service_metrics_2026-05-05.csv'

required_paths = [GEOMETRY_PATH, SPEED_PATH, FREQUENCY_PATH]
missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    missing_list = '\n'.join(f'- {path}' for path in missing_paths)
    raise FileNotFoundError(
        'Notebook 5 needs the output CSVs from Notebooks 2, 3, and 4. Missing:\n' + missing_list
    )

print(f'Project root: {PROJECT_ROOT.resolve()}')
print(f'Outputs:      {OUTPUT_DIR.resolve()}')

## 3. Load route-level metric tables

In [ ]:
geometry = pd.read_csv(GEOMETRY_PATH)
speed = pd.read_csv(SPEED_PATH)
frequency = pd.read_csv(FREQUENCY_PATH)

print(f'Geometry routes:  {len(geometry)}')
print(f'Speed routes:     {len(speed)}')
print(f'Frequency routes: {len(frequency)}')

## 4. Merge the metric tables

Each input table has one row per route. We keep the route identifiers once and suffix overlapping metric columns so their source stays clear.

In [ ]:
route_keys = ['route_id', 'route_short_name', 'route_long_name']

final_metrics = (
    geometry
    .merge(
        speed,
        on=route_keys,
        how='inner',
        suffixes=('_geometry', '_speed'),
        validate='one_to_one'
    )
    .merge(
        frequency,
        on=route_keys,
        how='inner',
        suffixes=('', '_frequency'),
        validate='one_to_one'
    )
)

print(f'Merged routes: {len(final_metrics)}')
final_metrics.head()

## 5. Define scoring helpers

Scores are converted to a 0-100 scale using percentile ranks.

- For metrics where higher is better, the best route gets a higher score.
- For metrics where lower is better, the direction is reversed.
- Missing values are filled with a neutral score of 50 so they do not dominate the final ranking.

In [ ]:
def percentile_score(series: pd.Series, higher_is_better: bool = True, neutral: float = 50) -> pd.Series:
    """Convert a numeric series to a 0-100 percentile score."""
    numeric = pd.to_numeric(series, errors='coerce')
    score = numeric.rank(pct=True, method='average') * 100
    if not higher_is_better:
        score = 101 - score
    return score.fillna(neutral).clip(lower=0, upper=100)


def weighted_score(df: pd.DataFrame, weights: dict) -> pd.Series:
    """Calculate a weighted score from score columns whose weights sum to 1."""
    total_weight = sum(weights.values())
    if not np.isclose(total_weight, 1.0):
        raise ValueError(f'Weights must sum to 1. Current sum: {total_weight}')
    result = pd.Series(0.0, index=df.index)
    for column, weight in weights.items():
        result += df[column] * weight
    return result

## 6. Prepare scoring inputs

A few metrics need interpretation before scoring:

- Loop routes can have very low straight-line endpoint distance, making directness mathematically misleading. These routes receive a neutral directness score and a loop flag.
- Stop density is scored as an efficiency metric, where fewer stops per km usually means fewer delays. This should be interpreted carefully because stops also provide access.
- Frequency combines median headway and total scheduled trips so limited special-purpose routes do not rank highly just because a few trips are close together.

In [ ]:
scored = final_metrics.copy()

scored['loop_or_circulator_flag'] = np.where(scored['avg_straight_line_km'] < 1, 1, 0)
scored['directness_ratio_for_scoring'] = scored['avg_directness_ratio'].where(
    scored['loop_or_circulator_flag'] == 0,
    np.nan
)

scored['speed_score'] = percentile_score(scored['avg_scheduled_speed_kmh'], higher_is_better=True)
scored['headway_score'] = percentile_score(scored['median_headway_min'], higher_is_better=False)
scored['trip_volume_score'] = percentile_score(scored['scheduled_trips'], higher_is_better=True)
scored['frequency_score'] = 0.70 * scored['headway_score'] + 0.30 * scored['trip_volume_score']
scored['service_span_score'] = percentile_score(scored['full_service_span_hours'], higher_is_better=True)
scored['directness_score'] = percentile_score(scored['directness_ratio_for_scoring'], higher_is_better=True)
scored.loc[scored['loop_or_circulator_flag'] == 1, 'directness_score'] = 50
scored['stop_spacing_efficiency_score'] = percentile_score(scored['stops_per_km'], higher_is_better=False)
scored['directional_balance_score'] = percentile_score(scored['directional_balance_ratio'], higher_is_better=True)
scored['all_day_service_score'] = scored['all_day_service_flag'].fillna(0) * 100

score_columns = [
    'speed_score', 'frequency_score', 'service_span_score', 'directness_score',
    'stop_spacing_efficiency_score', 'directional_balance_score', 'all_day_service_score'
]

scored[score_columns].describe().round(2)

## 7. Calculate the final score

The default weighting emphasizes rider usefulness and operating performance:

- **25% scheduled speed**
- **25% frequency**
- **15% service span**
- **15% route directness**
- **10% stop-spacing efficiency**
- **10% directional balance**

The all-day flag is kept as a diagnostic metric, but not directly included in the weighted score because service span and frequency already capture much of that concept.

In [ ]:
efficiency_weights = {
    'speed_score': 0.25,
    'frequency_score': 0.25,
    'service_span_score': 0.15,
    'directness_score': 0.15,
    'stop_spacing_efficiency_score': 0.10,
    'directional_balance_score': 0.10,
}

scored['final_efficiency_score'] = weighted_score(scored, efficiency_weights).round(2)

scored['efficiency_rank'] = scored['final_efficiency_score'].rank(
    ascending=False,
    method='min'
).astype(int)

scored['score_band'] = pd.cut(
    scored['final_efficiency_score'],
    bins=[-np.inf, 40, 55, 70, 85, np.inf],
    labels=['very low', 'low', 'moderate', 'high', 'very high']
)

scored = scored.sort_values(['efficiency_rank', 'route_short_name'])

scored[['efficiency_rank', 'route_short_name', 'route_long_name', 'final_efficiency_score', 'score_band']].head(20)

## 8. Create an advocacy priority lens

A route can be important even if its efficiency score is low. This lens highlights routes with many daily trips but weaker efficiency indicators. These are useful candidates for closer advocacy review because improvements could affect many riders.

In [ ]:
scored['service_importance_score'] = percentile_score(scored['scheduled_trips'], higher_is_better=True)
scored['improvement_need_score'] = 100 - scored['final_efficiency_score']
scored['advocacy_priority_score'] = (
    0.60 * scored['improvement_need_score'] +
    0.40 * scored['service_importance_score']
).round(2)

scored['advocacy_priority_rank'] = scored['advocacy_priority_score'].rank(
    ascending=False,
    method='min'
).astype(int)

priority_routes = scored.sort_values('advocacy_priority_rank')[[
    'advocacy_priority_rank', 'route_short_name', 'route_long_name',
    'advocacy_priority_score', 'final_efficiency_score', 'scheduled_trips',
    'avg_scheduled_speed_kmh', 'median_headway_min', 'avg_directness_ratio',
    'stops_per_km', 'full_service_span_hours'
]].head(20)

priority_routes

## 9. Final route ranking table

This table keeps the most important raw metrics beside the scoring outputs so the ranking remains explainable.

In [ ]:
final_columns = [
    'efficiency_rank', 'route_id', 'route_short_name', 'route_long_name',
    'final_efficiency_score', 'score_band', 'advocacy_priority_rank', 'advocacy_priority_score',
    'scheduled_trips', 'directions', 'all_day_service_flag',
    'avg_scheduled_speed_kmh', 'median_scheduled_speed_kmh', 'avg_duration_min',
    'avg_shape_path_km', 'avg_directness_ratio', 'loop_or_circulator_flag',
    'unique_stops', 'stops_per_km',
    'first_departure', 'last_departure', 'full_service_span_hours',
    'median_headway_min', 'p90_headway_min', 'directional_balance_ratio',
    'speed_score', 'frequency_score', 'service_span_score', 'directness_score',
    'stop_spacing_efficiency_score', 'directional_balance_score'
]

final_route_ranking = scored[final_columns].copy()

display_cols = [
    'efficiency_rank', 'route_short_name', 'route_long_name', 'final_efficiency_score',
    'score_band', 'scheduled_trips', 'avg_scheduled_speed_kmh', 'median_headway_min',
    'full_service_span_hours', 'directional_balance_ratio'
]

final_route_ranking[display_cols].head(25)

## 10. Highest-scoring routes

In [ ]:
top_efficiency_routes = final_route_ranking.sort_values('efficiency_rank').head(15)

top_efficiency_routes[[
    'efficiency_rank', 'route_short_name', 'route_long_name', 'final_efficiency_score',
    'scheduled_trips', 'avg_scheduled_speed_kmh', 'median_headway_min',
    'full_service_span_hours', 'avg_directness_ratio'
]]

## 11. Lowest-scoring routes

Low scores should be read as a starting point for investigation, not as proof that a route should be cut. Some routes may be school routes, coverage routes, loop routes, or special-purpose services.

In [ ]:
lowest_efficiency_routes = final_route_ranking.sort_values('efficiency_rank', ascending=False).head(15)

lowest_efficiency_routes[[
    'efficiency_rank', 'route_short_name', 'route_long_name', 'final_efficiency_score',
    'scheduled_trips', 'avg_scheduled_speed_kmh', 'median_headway_min',
    'full_service_span_hours', 'loop_or_circulator_flag'
]]

## 12. Routes most worth reviewing for improvements

In [ ]:
scored.sort_values('advocacy_priority_rank')[[
    'advocacy_priority_rank', 'route_short_name', 'route_long_name',
    'advocacy_priority_score', 'final_efficiency_score', 'scheduled_trips',
    'avg_scheduled_speed_kmh', 'median_headway_min', 'stops_per_km',
    'directional_balance_ratio'
]].head(15)

## 13. Optional charts

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_data = top_efficiency_routes.sort_values('final_efficiency_score')
    ax.barh(
        plot_data['route_short_name'].astype(str) + ' - ' + plot_data['route_long_name'].astype(str),
        plot_data['final_efficiency_score'],
        color='#2a9d8f'
    )
    ax.set_title('Highest-scoring MiWay routes by final efficiency score')
    ax.set_xlabel('Final efficiency score')
    ax.set_ylabel('Route')
    plt.tight_layout()
else:
    print('Install matplotlib to render this chart.')

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(9, 6))
    scatter = ax.scatter(
        scored['avg_scheduled_speed_kmh'],
        scored['median_headway_min'],
        c=scored['final_efficiency_score'],
        s=np.clip(scored['scheduled_trips'], 20, 260),
        cmap='viridis',
        alpha=0.80
    )
    ax.set_title('Speed, frequency, and final efficiency score')
    ax.set_xlabel('Average scheduled speed (km/h)')
    ax.set_ylabel('Median headway (minutes, lower is better)')
    fig.colorbar(scatter, ax=ax, label='Final efficiency score')
    plt.tight_layout()
else:
    print('Install matplotlib to render this chart.')

## 14. Sanity checks

In [ ]:
sanity_checks = {
    'routes in final ranking': len(final_route_ranking),
    'routes missing final score': final_route_ranking['final_efficiency_score'].isna().sum(),
    'minimum final score': final_route_ranking['final_efficiency_score'].min(),
    'maximum final score': final_route_ranking['final_efficiency_score'].max(),
    'loop/circulator routes with neutralized directness': int(final_route_ranking['loop_or_circulator_flag'].sum()),
    'routes marked all-day service': int(final_route_ranking['all_day_service_flag'].sum()),
    'highest ranked route': final_route_ranking.sort_values('efficiency_rank').iloc[0]['route_long_name'],
}

pd.Series(sanity_checks, name='value').to_frame()

## 15. Export final outputs

In [ ]:
final_ranking_path = OUTPUT_DIR / 'final_route_efficiency_ranking_2026-05-05.csv'
priority_routes_path = OUTPUT_DIR / 'advocacy_priority_routes_2026-05-05.csv'
score_components_path = OUTPUT_DIR / 'route_efficiency_score_components_2026-05-05.csv'

final_route_ranking.to_csv(final_ranking_path, index=False)
priority_routes.to_csv(priority_routes_path, index=False)
scored.to_csv(score_components_path, index=False)

print('Exported:')
print('-', final_ranking_path)
print('-', priority_routes_path)
print('-', score_components_path)

# What this notebook accomplished

Notebook 5 created the first complete MiWay route efficiency ranking for the representative weekday.

It produced:

- a final route efficiency score,
- route ranks,
- score bands,
- component scores,
- a separate advocacy priority lens,
- and final CSV outputs for reporting.

## Important interpretation note

This ranking is a structured screening tool. It should be used to identify routes worth discussing, not to make service decisions by itself. A low-scoring route may still be socially important if it serves students, shift workers, lower-density areas, seniors, or places with few alternatives.